In [1]:
# Save mortality by year and for each ensemble

In [1]:
import xarray as xr
import numpy as np
from utils.mortality_utils import att_frac
from utils.mortality_utils import mortality

In [3]:
# === Path config ===
BMR_DIR = "/glade/work/awells/air_quality/BMR/"
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"
O3_DIR = "/glade/work/awells/air_quality/CESM/ozone/OSDMA8_BC/"

In [4]:
# === Load data ===
BMR = xr.open_dataarray(f"{BMR_DIR}GBD_BMR_Country_Mask_popgrid_COPD_1990-2009.nc")  # three quantiles
population = xr.open_dataarray(f"{POP_DIR}ssp2_total_regrid_annual_2000-2100.nc")

In [5]:
# === Calculate beta for RR GBD 2021 ===

# Relative risk for 10ppb increase in OSDMA8, GBD 2021
RR_10ppb = 1.074  # [95% CI 1.014 – 1.137]
# equation is: RR = e^(beta*(x-TMREL)) where RR_10ppb = e^(10beta)
beta = np.log(RR_10ppb)/10

# TMREL from GBD 2021
TMREL = 32.4  # [95% CI 29.1 – 35.7]

In [ ]:
# === Path config ===
SAVE_DIR = "/glade/work/awells/air_quality/CESM/mortality/"
SCENARIOS = ["ARISE", "SSP245"]

# === Main loop ===
for scenario in SCENARIOS:
    for ens_num in range(1, 11):
        print(f"Processing {scenario}, Ensemble {ens_num:02d}")
        if scenario == "ARISE":
            dates = "2035-2068"
        elif scenario == "SSP245":
            dates = "2020-2068"

        o3 = xr.open_dataarray(f"{O3_DIR}OSDMA8_BC_popgrid_CESM2_{scenario}_{ens_num:02d}_{dates}.nc")

        M = []

        for year in o3["year"].values:
            AF = att_frac(o3.sel(year=year), TMREL, beta)
            POP = population.sel(year=year)
            mortality_year = mortality(AF, BMR, POP)
            M.append(mortality_year)

            mean_mortality = mortality_year.sum(dim=("lat", "lon")).sel(quantile="mean").round().values
            print(f"Mean mortality rate for {year} is {mean_mortality}")

        M_cleaned = [da.drop_vars("year", errors="ignore") for da in M]
        mortality_timeseries = xr.concat(M_cleaned, dim=(xr.DataArray(o3["year"].values, dims="year", name="year")))

        print(f"Saving mortality timeseries to {SAVE_DIR}")
        mortality_timeseries.to_netcdf(f"{SAVE_DIR}Mortality_CESM2_{scenario}_{ens_num:02d}_{dates}.nc")

Processing SSP245, Ensemble 10
Mean mortality rate for 2020 is 409893.0
Mean mortality rate for 2021 is 411703.0
Mean mortality rate for 2022 is 397607.0
Mean mortality rate for 2023 is 401446.0
Mean mortality rate for 2024 is 418294.0
Mean mortality rate for 2025 is 417250.0
Mean mortality rate for 2026 is 439179.0
Mean mortality rate for 2027 is 407337.0
Mean mortality rate for 2028 is 425289.0
Mean mortality rate for 2029 is 427066.0
Mean mortality rate for 2030 is 420397.0
Mean mortality rate for 2031 is 437112.0
Mean mortality rate for 2032 is 429269.0
Mean mortality rate for 2033 is 453742.0
Mean mortality rate for 2034 is 424160.0
Mean mortality rate for 2035 is 461722.0
Mean mortality rate for 2036 is 435603.0
Mean mortality rate for 2037 is 458293.0
Mean mortality rate for 2038 is 457624.0
Mean mortality rate for 2039 is 447905.0
Mean mortality rate for 2040 is 449950.0


HDF5-DIAG: Error detected in HDF5 (1.14.4-3) MPI-process 0:
  #000: H5F.c line 982 in H5Fflush(): unable to synchronously flush file
    major: File accessibility
    minor: Unable to flush data from cache
  #001: H5F.c line 955 in H5F__flush_api_common(): unable to flush file
    major: File accessibility
    minor: Unable to flush data from cache
  #002: H5VLcallback.c line 4098 in H5VL_file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #003: H5VLcallback.c line 4033 in H5VL__file_specific(): file specific failed
    major: Virtual Object Layer
    minor: Can't operate on object
  #004: H5VLnative_file.c line 312 in H5VL__native_file_specific(): unable to flush mounted file hierarchy
    major: File accessibility
    minor: Unable to flush data from cache
  #005: H5Fmount.c line 547 in H5F_flush_mounts(): unable to flush mounted file hierarchy
    major: File accessibility
    minor: Unable to flush data from cache
  #006: H5Fmo

Mean mortality rate for 2041 is 455104.0
Mean mortality rate for 2042 is 465165.0
Mean mortality rate for 2043 is 478276.0
Mean mortality rate for 2044 is 449267.0
Mean mortality rate for 2045 is 418182.0
Mean mortality rate for 2046 is 453993.0
Mean mortality rate for 2047 is 461300.0
Mean mortality rate for 2048 is 445678.0
Mean mortality rate for 2049 is 455461.0
Mean mortality rate for 2050 is 444556.0
Mean mortality rate for 2051 is 444379.0
Mean mortality rate for 2052 is 431370.0
Mean mortality rate for 2053 is 407453.0
Mean mortality rate for 2054 is 442903.0
Mean mortality rate for 2055 is 434649.0
Mean mortality rate for 2056 is 433375.0
Mean mortality rate for 2057 is 438601.0
Mean mortality rate for 2058 is 401007.0
Mean mortality rate for 2059 is 410394.0
Mean mortality rate for 2060 is 410953.0
Mean mortality rate for 2061 is 424648.0
Mean mortality rate for 2062 is 402398.0
Mean mortality rate for 2063 is 391646.0
Mean mortality rate for 2064 is 409132.0
Mean mortality r